In [6]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# -----------------------------
# Model Definition
# -----------------------------
class RobotLSTM(nn.Module):
    def __init__(self, input_size=18, hidden_size=128, output_size=6, num_layers=2):
        super(RobotLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out)
        return out


def time_series_plots(df, feature_type_lst, error=None):
    
    colors = px.colors.qualitative.Dark24[:6]

    # Create subplots for X, Y, Z
    fig = make_subplots(
        rows=6, cols=1,
        shared_xaxes=True,
        subplot_titles=(feature_type_lst),
        vertical_spacing=0.08
    )

    # Loop through each axis
    for idx, feature_type in enumerate(feature_type_lst, start=1):
        color = colors[idx - 1]

        # Plot original position
        if feature_type in df.columns:
            fig.add_trace(
                go.Scatter(
                    x=df['time'],
                    y=df[feature_type],
                    name=f"{feature_type.upper()} (true)",
                    mode='lines',
                    line=dict(color=color, width=2),
                ),
                row=idx, col=1
            )

        # Plot reconstructed / predicted position
        hat_col = f"pred_{feature_type}"
        if hat_col in df.columns:
            fig.add_trace(
                go.Scatter(
                    x=df['time'],
                    y=df[hat_col],
                    name=f"{feature_type.upper()} (pred)",
                    mode='lines',
                    line=dict(color=color, width=2, dash='dash')
                ),
                row=idx, col=1
            )

    # Optional: error shading or line
    if error is not None and "time" in df.columns:
        fig.add_trace(
            go.Scatter(
                x=df['time'],
                y=error,
                name="Error",
                mode='lines',
                line=dict(color='red', dash='dot')
            ),
            row=3, col=1
        )

    # Axis titles
    fig.update_xaxes(title_text="Time (s)", row=6, col=1, rangeslider_visible=True)
    for i, feature_type in enumerate(feature_type_lst, start=1):
        fig.update_yaxes(title_text=f"{feature_type.upper()} (m)", row=i, col=1)

    # Layout
    fig.update_layout(
        height=1000,
        title_text="Feature Reconstruction via LSTM",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, xanchor="right", x=1)
    )

    return fig


def data_prep(df, input_lst, output_type_name, is_target_known=True):
    # -----------------------------
    # Normalization Setup
    # -----------------------------
    input_scaler = StandardScaler()
    output_scaler = None

    output_lst = [f'{output_type_name}{i}' for i in range(6)]
    input_size = len(input_lst)
    output_size = len(output_lst)
    seq_len = 50

    # Fit on all data (you can also fit only on train split for better generalization)
    X_scaled = input_scaler.fit_transform(df[input_lst])
    num_samples = len(X_scaled) // seq_len

    y_tensor = None
    if is_target_known:
        output_scaler = StandardScaler()
        y_scaled = output_scaler.fit_transform(df[output_lst])
        y = y_scaled[:num_samples * seq_len].reshape(num_samples, seq_len, output_size)
        y_tensor = torch.tensor(y, dtype=torch.float32)

    # -----------------------------
    # Reshape to Sequences
    # -----------------------------
    X = X_scaled[:num_samples * seq_len].reshape(num_samples, seq_len, input_size)

    # Convert to float tensors
    X_tensor = torch.tensor(X, dtype=torch.float32)

    return X_scaled, X_tensor, y_tensor, output_scaler


def prediction_df(X_scaled, output_scaler, output_lst, seq_len, input_size, model_name):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    loaded_model = RobotLSTM(input_size, 128, len(output_lst), 2)
    loaded_model.load_state_dict(torch.load(model_name, map_location=device))
    loaded_model.to(device)
    loaded_model.eval()

    # -----------------------------
    # Prepare Data as Sequential Input
    # -----------------------------

    # Ensure data is reshaped as (num_samples, seq_len, input_size)
    num_samples = len(X_scaled) // seq_len
    X_seq = X_scaled[:num_samples * seq_len].reshape(num_samples, seq_len, input_size)

    # Convert to tensor
    X_seq_tensor = torch.tensor(X_seq, dtype=torch.float32).to(device)

    # -----------------------------
    # Run Inference 
    # -----------------------------
    predictions_scaled = []

    with torch.no_grad():
        for batch in X_seq_tensor:
            batch = batch.unsqueeze(0)  # shape (1, seq_len, input_size)
            pred = loaded_model(batch)  # shape (1, seq_len, output_size)
            predictions_scaled.append(pred.squeeze(0).cpu().numpy())

    # Stack all batches into one (num_samples * seq_len, output_size)
    predictions_scaled = np.vstack(predictions_scaled)

    # -----------------------------
    # Denormalize Predictions
    # -----------------------------
    predictions_original = output_scaler.inverse_transform(predictions_scaled)

    # -----------------------------
    # Save Predictions to DataFrame
    # -----------------------------
    pred_cols = [f'pred_{name}' for name in output_lst]
    pred_df = pd.DataFrame(predictions_original, columns=pred_cols)

    # Trim df to match predictions (in case of leftover rows)
    # df_pred = df.iloc[:len(pred_df)].copy()
    # df_pred[pred_cols] = pred_df

    return pred_df



In [ ]:
def make_prediction_df(df, input_lst, output_type, model_name, df_out_name):
    output_lst = [f'{output_type}{i}' for i in range(6)]

    X_scaled, X_tensor, _, output_scaler = data_prep(df, input_lst, output_type, is_target_known=True)

    df_pred = prediction_df(X_scaled, output_scaler, output_lst, 50, len(input_lst), model_name)
    df_pred = pd.concat([df_pred.reset_index(drop=True), df['time'].reset_index(drop=True)], axis=1)
    df_pred = pd.concat([df_pred.reset_index(drop=True), df[output_lst].reset_index(drop=True)], axis=1)

    # output_lst = [f'pred_{output_type}{i}' for i in range(6)]

    df_pred[::100].to_feather(df_out_name)

    fig = time_series_plots(df_pred.iloc[::100], output_lst)
    fig.show()


In [8]:
import pandas as pd

# df = pd.read_feather('../data/aursad/aursad_training.feather')
input_lst = ['q0','q1','q2','q3','q4','q5', 'x', 'y', 'z']
output_type = 'Current'
model_name = '../models/current_pred_lstm100.pt'
df_out_name = 'current_pred_test.feather'

make_prediction_df(df, input_lst, output_type, model_name, df_out_name)

AttributeError: 'NoneType' object has no attribute 'inverse_transform'

In [ ]:
from sklearn.preprocessing import StandardScaler
import pandas as pd
import torch

df_aursad = pd.read_feather('../data/aursad/aursad_training.feather')
df_rad = pd.read_feather('../data/rad/rad_add_current.feather')
input_lst = ['q0','q1','q2','q3','q4','q5', 'x', 'y', 'z']
output_type_name = 'Current'
output_lst = [f'{output_type_name}{i}' for i in range(6)]

X_scaled, X_tensor, _, _ = data_prep(df_rad, input_lst, output_type_name, is_target_known=False)

output_scaler = StandardScaler()
y_scaled = output_scaler.fit_transform(df_aursad[output_lst])

df_pred = prediction_df(X_scaled, output_scaler, 50, 
                        len(input_lst), len(output_lst), 
                        '../models/speed_pred_lstm30.pt')
df_pred = pd.concat([df_pred.reset_index(drop=True), df_rad['time'].reset_index(drop=True)], axis=1)

output_lst = [f'pred_{output_type_name}{i}' for i in range(6)]
fig = time_series_plots(df_pred.iloc[::100], output_lst)
fig.show()


In [ ]:
df_aursad = pd.read_feather('../data/aursad/aursad_training.feather')
df_rad = pd.read_feather('../data/rad/rad_add_current.feather')
input_lst = ['q0','q1','q2','q3','q4','q5', 'x', 'y', 'z', 
            'Current0', 'Current1', 'Current2', 'Current3', 'Current4', 'Current5']
output_type_name = 'Speed'
output_lst = [f'{output_type_name}{i}' for i in range(6)]

X_scaled, X_tensor, _, _ = data_prep(df_rad, input_lst, output_type_name, is_target_known=False)

output_scaler = StandardScaler()
y_scaled = output_scaler.fit_transform(df_aursad[output_lst])

df_pred = prediction_df(X_scaled, output_scaler, 50, 
                        len(input_lst), len(output_lst), 
                        '../models/speed_pred_lstm30.pt')
df_pred = pd.concat([df_pred.reset_index(drop=True), df_rad['time'].reset_index(drop=True)], axis=1)

output_lst = [f'pred_{output_type_name}{i}' for i in range(6)]
fig = time_series_plots(df_pred.iloc[::100], output_lst)
fig.show()


In [ ]:
# df = pd.read_feather('../data/aursad/aursad_training.feather')
input_lst = ['q0','q1','q2','q3','q4','q5', 'x', 'y', 'z', 'time',
            'Current0', 'Current1', 'Current2', 'Current3', 'Current4', 'Current5',
            'Speed0','Speed1','Speed2','Speed3','Speed4','Speed5']
output_type_name = 'Temperature'
output_size = 6
input_size = len(input_lst)


X_scaled, X_tensor, y_tensor, output_scaler = data_prep(df, input_lst, output_type_name, is_target_known=True)

output_lst = [f'{output_type_name}{i}' for i in range(6)]

df_pred = prediction_df(X_scaled, output_scaler, 50, 
                        len(input_lst), len(output_lst), 
                        '../models/temp_pred_lstm30.pt')       
df_pred = pd.concat([df_pred.reset_index(drop=True), df[['time'] + output_lst].reset_index(drop=True)], axis=1)



fig = time_series_plots(df_pred.iloc[::100], output_lst)
fig.show()

